***
# Clean NOAA weather observations
The processed NOAA LCD weather observations were cleaned by handling missing values, converting trace precipitation (T) to a small numeric value, engineering binary indicators from METAR weather codes, transforming wind direction into Cartesian components (WindX and WindY), and removing metadata and low-value fields. The resulting dataset contains only weather features relevant to flight delay prediction and is ready to be merged with the BTS flight data.
***

***
## Select the station-year input

`YEAR` and `AIRPORT` identify the processed NOAA file created by the preceding processing notebook. The cleaning operations below assume that this file contains one airport's observations in chronological row order.
***

In [226]:
YEAR = 2019

# AIRPORT = "JFK"
AIRPORT = "ORD"
# AIRPORT = "ATL"

PROCESSED_DATA_FILE = f"../data/noaa/processed/{AIRPORT}_{YEAR}.csv"
CLEAN_DATA_FILE = f"../data/noaa/cleaned/{AIRPORT}_{YEAR}.csv"

In [227]:
import pandas as pd
import numpy as np

In [228]:
df = pd.read_csv(PROCESSED_DATA_FILE)

***
## Normalize hourly precipitation

NOAA uses `T` to report a measurable trace amount below the normal reporting precision. Both observed trace formats are mapped to `0.001` so the model can distinguish a trace from no precipitation. Any other nonnumeric values are converted to `NaN` for explicit handling later.
***

In [229]:
df["HourlyPrecipitation"] = (
    df["HourlyPrecipitation"]
        .replace("T", 0.001)
        .replace("0.00T", 0.001)
)

df["HourlyPrecipitation"] = pd.to_numeric(
    df["HourlyPrecipitation"],
    errors="coerce"
)

***
## Decode present-weather conditions

The NOAA present-weather field contains compact METAR-style codes. Missing reports are converted to empty strings and the text is normalized to uppercase before producing binary indicators for rain, drizzle, snow, fog, mist, thunderstorms, freezing precipitation, and showers. Multiple indicators may be active for the same observation.

`PrecipOccurred` combines the numeric precipitation measurement with the rain, snow, and drizzle indicators. This retains evidence of precipitation when either the measured amount or the reported weather code is available.
***

In [230]:
weather = (
    df["HourlyPresentWeatherType"]
        .fillna("")
        .str.upper()
)

In [231]:
df["Rain"] = weather.str.contains(r"\bRA\b", regex=True).astype(int)

df["Drizzle"] = weather.str.contains(r"\bDZ\b", regex=True).astype(int)

df["Snow"] = weather.str.contains(r"\bSN\b", regex=True).astype(int)

df["Fog"] = weather.str.contains(r"\bFG\b", regex=True).astype(int)

df["Mist"] = weather.str.contains(r"\bBR\b", regex=True).astype(int)

df["Thunderstorm"] = weather.str.contains(r"\bTS\b", regex=True).astype(int)

df["FreezingPrecip"] = weather.str.contains(r"FZ", regex=True).astype(int)

df["Showers"] = weather.str.contains(r"\bSH\b", regex=True).astype(int)

In [232]:
df["PrecipOccurred"] = (
    (df["HourlyPrecipitation"].fillna(0) > 0) |
    (df["Rain"] == 1) |
    (df["Snow"] == 1) |
    (df["Drizzle"] == 1)
).astype(int)

***
## Engineer Wind Components (WindX and WindY)

Wind direction is a **circular** variable, where 0° and 360° represent the same direction. Using the raw direction as a numeric feature introduces an artificial discontinuity; for example, 359° and 1° are nearly identical meteorologically but appear numerically far apart.

To preserve the directional information, wind direction is transformed into two Cartesian components:

- **WindX** – East-West component of the wind vector
- **WindY** – North-South component of the wind vector

Wind speed is interpolated first. Missing directions are filled using unit-circle components so values near 0° and 360° remain close together. The direction components are normalized before the wind speed is applied, which keeps the final vector length equal to `HourlyWindSpeed`.

Because NOAA reports the direction the wind comes **from**, conventional east-west and north-south components are calculated as:

- `WindX = -WindSpeed × sin(WindDirection)`
- `WindY = -WindSpeed × cos(WindDirection)`

Positive `WindX` points east and positive `WindY` points north. This representation preserves the circular nature of wind direction and is more suitable for machine learning than raw degrees. After creating these features, the original `HourlyWindDirection` column is no longer needed.
***

In [233]:
df["HourlyWindDirection"] = pd.to_numeric(
    df["HourlyWindDirection"],
    errors="coerce"
)

df["HourlyWindSpeed"] = (
    pd.to_numeric(df["HourlyWindSpeed"], errors="coerce")
        .interpolate(method="linear", limit_direction="both")
)

radians = np.deg2rad(df["HourlyWindDirection"])

direction_x = pd.Series(-np.sin(radians), index=df.index).interpolate(
    method="linear",
    limit_direction="both"
)
direction_y = pd.Series(-np.cos(radians), index=df.index).interpolate(
    method="linear",
    limit_direction="both"
)

direction_length = np.hypot(direction_x, direction_y)
direction_x = direction_x / direction_length
direction_y = direction_y / direction_length

df["WindX"] = df["HourlyWindSpeed"] * direction_x
df["WindY"] = df["HourlyWindSpeed"] * direction_y

***
## Resolve precipitation missingness and remove source fields

A missing precipitation amount is set to zero only when the combined weather evidence says that no precipitation occurred. Missing amounts associated with reported precipitation remain missing so they can be interpolated rather than incorrectly treated as dry conditions.

After the weather indicators and wind components have been derived, the original coded weather and wind-direction columns are removed to avoid carrying redundant, model-unfriendly source fields.
***

In [234]:
mask = (
    df["HourlyPrecipitation"].isna() &
    (df["PrecipOccurred"] == 0)
)

df.loc[mask, "HourlyPrecipitation"] = 0

In [235]:
df.drop(columns=["HourlyPresentWeatherType"], inplace=True)
df.drop(columns=["HourlyWindDirection"], inplace=True)

***
## Impute continuous weather measurements

Hourly precipitation and the retained continuous weather fields are coerced to numeric values and filled by linear interpolation. `limit_direction="both"` also fills missing values at the beginning or end using the nearest available observation.

> **Ordering assumption:** pandas interpolates in the dataframe's current row order. Confirm that rows are sorted chronologically before this step. For model evaluation, fit or apply time-dependent preprocessing within each training fold to avoid information from future or held-out observations influencing earlier values.
***

In [236]:
df["HourlyPrecipitation"] = (
    df["HourlyPrecipitation"]
        .interpolate(method="linear", limit_direction="both")
)

In [237]:
cols = [
    "HourlyDewPointTemperature",
    "HourlyDryBulbTemperature",
    "HourlyRelativeHumidity",
    "HourlyVisibility",
    "HourlyWindSpeed"
]

df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

df[cols] = (
    df[cols]
        .interpolate(method="linear", limit_direction="both")
)

***
## Parse observation timestamps

Convert `DATE` from CSV text to pandas `datetime64` values so it can be used for chronological sorting, time-based joins, and feature engineering. Unparseable values become `NaT` because `errors="coerce"`; their count should be checked before merging this table with flight records. A consistent timezone should also be established before matching weather observations to flight times.
***

In [238]:
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")

***
# Validate the cleaned NOAA data

Run read-only checks before saving the file. These cells create temporary validation objects but do not change, sort, filter, or add columns to `df`.
***

***
## Check the output columns

Confirm that every expected date, weather measurement, weather indicator, and wind component is present. Also report any extra columns for review.
***

In [239]:
expected_columns = [
    "DATE",
    "HourlyDewPointTemperature",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlyVisibility",
    "HourlyWindSpeed",
    "Rain",
    "Drizzle",
    "Snow",
    "Fog",
    "Mist",
    "Thunderstorm",
    "FreezingPrecip",
    "Showers",
    "PrecipOccurred",
    "WindX",
    "WindY"
]

schema_validation = pd.Series({
    "row count": len(df),
    "column count": len(df.columns),
    "missing expected columns": sorted(set(expected_columns) - set(df.columns)),
    "unexpected columns": sorted(set(df.columns) - set(expected_columns))
}, name="schema validation")

schema_validation

row count                   12876
column count                   18
missing expected columns       []
unexpected columns             []
Name: schema validation, dtype: object

***
## Check timestamps and row order

Check the `DATE` type, missing values, selected year, duplicate timestamps, and chronological order. NOAA may contain more than one report per hour, so the interval summary is shown for review without assuming exactly one row per hour.
***

In [240]:
date_validation = pd.Series({
    "DATE dtype": str(df["DATE"].dtype),
    "missing timestamps": int(df["DATE"].isna().sum()),
    "timestamps outside selected year": int(
        (df["DATE"].notna() & (df["DATE"].dt.year != YEAR)).sum()
    ),
    "first timestamp": df["DATE"].min(),
    "last timestamp": df["DATE"].max(),
    "timezone": str(df["DATE"].dt.tz),
    "fully duplicated rows": int(df.duplicated().sum()),
    "rows with duplicated timestamps": int(df["DATE"].duplicated(keep=False).sum()),
    "rows already in chronological order": bool(df["DATE"].is_monotonic_increasing)
}, name="timestamp validation")

date_validation

DATE dtype                                  datetime64[ns]
missing timestamps                                       0
timestamps outside selected year                         0
first timestamp                        2019-01-01 00:00:00
last timestamp                         2019-12-31 23:51:00
timezone                                              None
fully duplicated rows                                    1
rows with duplicated timestamps                         18
rows already in chronological order                   True
Name: timestamp validation, dtype: object

In [241]:
unique_timestamps = (
    df["DATE"].dropna().drop_duplicates().sort_values()
)
timestamp_intervals = unique_timestamps.diff().dropna()

timestamp_intervals.value_counts().head(10)

DATE
0 days 01:00:00    5970
0 days 00:09:00    1541
0 days 00:51:00    1208
0 days 00:02:00     384
0 days 00:07:00     154
0 days 00:08:00     138
0 days 00:10:00     129
0 days 00:11:00     120
0 days 00:13:00     117
0 days 00:58:00     116
Name: count, dtype: int64

***
## Check missing and numeric values

Count missing values in every column. Then confirm that each weather feature can be read as a number and contains no positive or negative infinity. These checks do not fill or replace any values.
***

In [242]:
missing_validation = pd.DataFrame({
    "missing count": df.isna().sum(),
    "missing percent": df.isna().mean().mul(100)
})

missing_validation

,missing count,missing percent
DATE,0,0.0
HourlyDewPointTemperature,0,0.0
HourlyDryBulbTemperature,0,0.0
HourlyPrecipitation,0,0.0
HourlyRelativeHumidity,0,0.0
HourlyVisibility,0,0.0
HourlyWindSpeed,0,0.0
Rain,0,0.0
Drizzle,0,0.0
Snow,0,0.0


In [243]:
numeric_columns = [column for column in expected_columns if column != "DATE"]
numeric_values = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

numeric_validation = pd.DataFrame({
    "source dtype": df[numeric_columns].dtypes.astype(str),
    "failed numeric conversion": [
        int((df[column].notna() & numeric_values[column].isna()).sum())
        for column in numeric_columns
    ],
    "infinite values": [
        int(numeric_values[column].isin([np.inf, -np.inf]).sum())
        for column in numeric_columns
    ]
}, index=numeric_columns)

numeric_validation

,source dtype,failed numeric conversion,infinite values
HourlyDewPointTemperature,float64,0,0
HourlyDryBulbTemperature,float64,0,0
HourlyPrecipitation,float64,0,0
HourlyRelativeHumidity,float64,0,0
HourlyVisibility,float64,0,0
HourlyWindSpeed,float64,0,0
Rain,int64,0,0
Drizzle,int64,0,0
Snow,int64,0,0
Fog,int64,0,0


***
## Check measurement ranges

Look for impossible negative precipitation, visibility, or wind speed values and relative humidity outside 0–100%. Display summary statistics so unusually large or small measurements can be reviewed rather than removed automatically.
***

In [244]:
range_validation = pd.Series({
    "negative precipitation values": int((numeric_values["HourlyPrecipitation"] < 0).sum()),
    "relative humidity outside 0-100": int(
        (~numeric_values["HourlyRelativeHumidity"].between(0, 100)).sum()
    ),
    "negative visibility values": int((numeric_values["HourlyVisibility"] < 0).sum()),
    "negative wind speed values": int((numeric_values["HourlyWindSpeed"] < 0).sum())
}, name="measurement range validation")

range_validation

negative precipitation values      0
relative humidity outside 0-100    0
negative visibility values         0
negative wind speed values         0
Name: measurement range validation, dtype: int64

In [245]:
numeric_values.describe().T

,count,mean,std,min,25%,50%,75%,max
HourlyDewPointTemperature,12876.0,40.063762,19.432090,-32.000000,26.000000,3.900000e+01,57.000000,77.000000
HourlyDryBulbTemperature,12876.0,48.764989,20.700372,-23.000000,33.000000,4.800000e+01,67.000000,95.000000
HourlyPrecipitation,12876.0,0.009600,0.048580,0.000000,0.000000,0.000000e+00,0.001000,1.090000
HourlyRelativeHumidity,12876.0,74.292016,16.511437,12.000000,63.000000,7.700000e+01,88.000000,100.000000
HourlyVisibility,12876.0,8.447594,2.792042,0.000000,8.000000,1.000000e+01,10.000000,10.000000
HourlyWindSpeed,12876.0,9.853604,5.271893,0.000000,6.000000,9.000000e+00,13.000000,45.000000
Rain,12876.0,0.118282,0.322954,0.000000,0.000000,0.000000e+00,0.000000,1.000000
Drizzle,12876.0,0.017086,0.129597,0.000000,0.000000,0.000000e+00,0.000000,1.000000
Snow,12876.0,0.072926,0.260026,0.000000,0.000000,0.000000e+00,0.000000,1.000000
Fog,12876.0,0.008310,0.090783,0.000000,0.000000,0.000000e+00,0.000000,1.000000


***
## Check engineered features

Confirm that weather indicators contain only 0 or 1. Compare `PrecipOccurred` with the precipitation amount and weather indicators, and compare the wind-vector length with hourly wind speed. Differences are reported for review without changing the features.
***

In [246]:
binary_columns = [
    "Rain", "Drizzle", "Snow", "Fog", "Mist",
    "Thunderstorm", "FreezingPrecip", "Showers", "PrecipOccurred"
]

binary_validation = pd.DataFrame({
    "missing values": numeric_values[binary_columns].isna().sum(),
    "values outside 0 or 1": [
        int((~numeric_values[column].isin([0, 1])).sum())
        for column in binary_columns
    ]
})

binary_validation

,missing values,values outside 0 or 1
Rain,0,0
Drizzle,0,0
Snow,0,0
Fog,0,0
Mist,0,0
Thunderstorm,0,0
FreezingPrecip,0,0
Showers,0,0
PrecipOccurred,0,0


In [247]:
expected_precip_occurred = (
    (numeric_values["HourlyPrecipitation"].fillna(0) > 0)
    | (numeric_values["Rain"] == 1)
    | (numeric_values["Snow"] == 1)
    | (numeric_values["Drizzle"] == 1)
).astype(int)

wind_magnitude = np.hypot(numeric_values["WindX"], numeric_values["WindY"])
wind_magnitude_difference = (
    wind_magnitude - numeric_values["HourlyWindSpeed"]
).abs()

engineered_feature_validation = pd.Series({
    "PrecipOccurred mismatches": int(
        (numeric_values["PrecipOccurred"] != expected_precip_occurred).sum()
    ),
    "wind magnitude differences above 0.001": int(
        (wind_magnitude_difference > 0.001).sum()
    ),
    "largest wind magnitude difference": wind_magnitude_difference.max()
}, name="engineered feature validation")

engineered_feature_validation

PrecipOccurred mismatches                 0.000000e+00
wind magnitude differences above 0.001    0.000000e+00
largest wind magnitude difference         3.552714e-15
Name: engineered feature validation, dtype: float64

***
## Review before saving

Review any nonzero validation counts and unusual numeric ranges before using the clean file for feature engineering. The existing save step remains last and is unchanged.
***

***
## Sort weather observations chronologically

Sort by `DATE` before saving so time-based joins and feature engineering receive observations in a consistent chronological order. A stable sort preserves the existing order of reports that share the same timestamp.
***

In [248]:
df = df.sort_values("DATE", kind="stable").reset_index(drop=True)

***
## Save the cleaned and validted weather data

Create the clean-data directory when needed and write the model-ready NOAA dataframe to `JDK.csv` without including the pandas index.
***

In [249]:
from pathlib import Path

Path(CLEAN_DATA_FILE).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_DATA_FILE, index=False)

In [250]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12876 entries, 0 to 12875
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   DATE                       12876 non-null  datetime64[ns]
 1   HourlyDewPointTemperature  12876 non-null  float64       
 2   HourlyDryBulbTemperature   12876 non-null  float64       
 3   HourlyPrecipitation        12876 non-null  float64       
 4   HourlyRelativeHumidity     12876 non-null  float64       
 5   HourlyVisibility           12876 non-null  float64       
 6   HourlyWindSpeed            12876 non-null  float64       
 7   Rain                       12876 non-null  int64         
 8   Drizzle                    12876 non-null  int64         
 9   Snow                       12876 non-null  int64         
 10  Fog                        12876 non-null  int64         
 11  Mist                       12876 non-null  int64         
 12  Thun

In [251]:
df.head()

,DATE,HourlyDewPointTemperature,HourlyDryBulbTemperature,HourlyPrecipitation,HourlyRelativeHumidity,HourlyVisibility,HourlyWindSpeed,Rain,Drizzle,Snow,Fog,Mist,Thunderstorm,FreezingPrecip,Showers,PrecipOccurred,WindX,WindY
0,2019-01-01 00:00:00,33.0,34.0,0.000,97.0,9.94,9.0,0,0,0,0,0,0,0,0,0,8.457234,-3.078181
1,2019-01-01 00:51:00,32.0,34.0,0.000,92.0,10.00,8.0,0,0,0,0,0,0,0,0,0,4.000000,-6.928203
2,2019-01-01 01:51:00,31.0,33.0,0.000,92.0,10.00,10.0,0,0,0,0,0,0,0,0,0,6.427876,-7.660444
3,2019-01-01 02:09:00,31.0,33.0,0.000,92.0,6.00,13.0,0,0,0,0,1,0,0,0,0,8.356239,-9.958578
4,2019-01-01 02:17:00,31.0,33.0,0.001,92.0,3.00,9.0,0,0,1,0,1,0,0,0,1,5.785088,-6.894400
